# Verify and read a Brazil Public Data Map release

This notebook downloads one public release from Kaggle, checks every file against its `release_manifest.json`, and reads a table. It uses only the public dataset and standard libraries plus `pandas` and `pyarrow`. It does not connect to any private system.

Set `DATASET` to any dataset from the catalogue, for example a PNCP subject dataset (`lucasrangelss/pncp-organizations-data`) or a per-table educational dataset.

In [ ]:
DATASET = "lucasrangelss/pncp-organizations-data"

In [ ]:
import hashlib, json, os
from pathlib import Path

import pandas as pd

kaggle_input = Path("/kaggle/input") / DATASET.split("/")[1]
if kaggle_input.exists():
    root = kaggle_input
else:
    import kagglehub  # pip install kagglehub
    root = Path(kagglehub.dataset_download(DATASET))
print("release directory:", root)

## 1. Verify every file against the manifest

In [ ]:
def sha256(path, block=8 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(block), b""):
            h.update(chunk)
    return h.hexdigest()

manifest = json.loads((root / "release_manifest.json").read_text(encoding="utf-8"))
bad = [f["path"] for f in manifest["files"]
       if not (root / f["path"]).is_file() or sha256(root / f["path"]) != f["sha256"]]
print("files listed:", len(manifest["files"]), "| mismatches:", bad or "none")
assert not bad, "release files do not match the manifest"

## 2. See what the release contains

In [ ]:
tables = manifest.get("tables")
if tables:  # subject dataset with several table-layer files
    catalogue = pd.DataFrame(tables)[["layer", "table_id", "row_count", "data_cutoff", "parquet_path"]]
else:       # single-table dataset
    catalogue = pd.DataFrame([{"layer": manifest["layer"], "table_id": manifest["table_id"],
                               "row_count": manifest["row_count"], "parquet_path": manifest["files"][0]["path"]}])
catalogue

## 3. Read one table

Parquet files keep their layer in the filename (`raw__`, `trusted__`, `semantic__`). Values are published as held in the snapshot; the manifest records the cutoff and row counts.

In [ ]:
first = catalogue.iloc[0]
df = pd.read_parquet(root / first["parquet_path"])
print(first["table_id"], first["layer"], "->", len(df), "rows,", len(df.columns), "columns")
assert len(df) == first["row_count"], "row count differs from the manifest"
df.head()

## Limits

- The manifest proves the files are the ones that were published. It does not prove the source data is complete or current; the PNCP source can change after the cutoff.
- A public source is not automatically free of personal data. Read the dataset card and source terms before reuse.
- See the [Brazil Public Data Map](https://github.com/LucasRangelSSouza/brazil-public-data-map) repository for the source registry and the release tooling.